# Sovereign Tier-3 Audio Collision Orchestrator

This notebook demonstrates the dual-actor Ray design for querying sound libraries:
1. **SongOrchestrator (Ray Actor):** Targets full tracks (`asset_type = 'SONG'`) and utilizes SQL filtering on `tempo` and `key` combined with LanceDB semantic search.
2. **SampleOrchestrator (Ray Actor):** Targets loop/one-shot samples (`asset_type = 'SAMPLE'`). Since samples lack tempo and key, it filters on `drum_type` (e.g., KICK, SNARE, BASS) and executes semantic search.

We also perform **Scikit-Learn Signature Analysis** using a Random Forest to evaluate which audio features (RMS, Crest, Spectral Centroid, etc.) are most characteristic of target styles.

In [1]:
import sys, os, time, duckdb, lancedb, pandas as pd

DUCKDB_PATH = r"C:\STUDIES_BACKUP\Legion-Jacked-Pipeline\ableton-session-intelligence\web_intel_sonicdb.duckdb"
LANCEDB_PATH = r"C:\STUDIES_BACKUP\Legion-Jacked-Pipeline\vectors\lancedb_store"

print("📊 LIBRARY STATS & PATHS")
print("-" * 40)
print(f"DuckDB Database : {DUCKDB_PATH}")
print(f"LanceDB Store   : {LANCEDB_PATH}")

conn = duckdb.connect(DUCKDB_PATH)
counts_df = conn.execute("""
    SELECT 
        asset_type,
        COUNT(*) as count,
        COUNT(tempo) as with_tempo,
        COUNT(key) as with_key
    FROM audio_features 
    GROUP BY asset_type
""").df()
print("\nData breakdown by asset_type:")
print(counts_df.to_string(index=False))

📊 LIBRARY STATS & PATHS
----------------------------------------
DuckDB Database : C:\STUDIES_BACKUP\Legion-Jacked-Pipeline\ableton-session-intelligence\web_intel_sonicdb.duckdb
LanceDB Store   : C:\STUDIES_BACKUP\Legion-Jacked-Pipeline\vectors\lancedb_store

Data breakdown by asset_type:
asset_type  count  with_tempo  with_key
 GENERATED      2           2         2
    SAMPLE    411           0         0
      SONG    671         671       671


In [2]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
import numpy as np

print("🌲 SCIKIT-LEARN SIGNATURE ANALYSIS (Random Forest)")
print("-" * 60)

# Extract audio physics features from DuckDB for classifier
ml_df = conn.execute("""
    SELECT 
        rms_db, crest_factor, sub_bass_energy, bass_energy, mid_energy, high_energy, spectral_centroid,
        CASE WHEN LOWER(filename) LIKE '%chris lake%' THEN 1 ELSE 0 END as is_target
    FROM audio_features
    WHERE rms_db IS NOT NULL
""").df()

features = ['rms_db', 'crest_factor', 'sub_bass_energy', 'bass_energy', 'mid_energy', 'high_energy', 'spectral_centroid']
X = ml_df[features]
y = ml_df['is_target']

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

rf = RandomForestClassifier(n_estimators=100, random_state=42, class_weight='balanced')
rf.fit(X_scaled, y)

importances = dict(zip(features, rf.feature_importances_))
print("Feature Importances for identifying 'Chris Lake' signature:")
for feat, imp in sorted(importances.items(), key=lambda x: x[1], reverse=True):
    print(f"  - {feat:<20}: {imp:.4f}")

# Close DuckDB connection in main process to release lock for Ray workers
conn.close()
print("\n🔒 DuckDB connection closed in main process.")

🌲 SCIKIT-LEARN SIGNATURE ANALYSIS (Random Forest)
------------------------------------------------------------
Feature Importances for identifying 'Chris Lake' signature:
  - bass_energy         : 0.1996
  - sub_bass_energy     : 0.1846
  - rms_db              : 0.1782
  - crest_factor        : 0.1529
  - spectral_centroid   : 0.0977
  - high_energy         : 0.0977
  - mid_energy          : 0.0893

🔒 DuckDB connection closed in main process.


In [2]:
import ray
from typing import List, Optional
from pydantic import BaseModel

# Schema Declarations
class Level3Query(BaseModel):
    bpm: Optional[float] = None
    key: Optional[str] = None
    drum_type: Optional[str] = None
    concept: str
    top_k: int = 3
    bpm_tolerance: float = 3.0

class TrackPhysics(BaseModel):
    filename: str
    dsp_tempo: Optional[float] = None
    dsp_key: Optional[str] = None
    rms_db: Optional[float] = None
    crest_factor: Optional[float] = None
    sub_bass_energy: Optional[float] = None

class SovereignDelta(BaseModel):
    filename: str
    rms_delta: float
    crest_delta: float
    sovereign_score: float
    grade: str

class CollisionRecord(BaseModel):
    filename: str
    physics: TrackPhysics
    delta: SovereignDelta
    vector_score: float
    source_level: int = 3

class CollisionResult(BaseModel):
    query_concept: str
    query_bpm: Optional[float]
    query_key: Optional[str]
    level: int
    records: List[CollisionRecord]
    total_found: int
    duck_ms: float
    lance_ms: float
    merge_ms: float

In [4]:
@ray.remote(num_cpus=1)
class SongOrchestrator:
    def __init__(self, duckdb_path: str, lancedb_path: str):
        import duckdb, lancedb
        from sentence_transformers import SentenceTransformer
        self.duck_conn = duckdb.connect(duckdb_path, read_only=True)
        self.lance_db = lancedb.connect(lancedb_path)
        self.table_name = "audio_vibe_gpu"
        self.embedder = SentenceTransformer("all-MiniLM-L6-v2")

    def run_collision_query(self, query: Level3Query) -> CollisionResult:
        import time
        sql_start = time.perf_counter()
        sql = "SELECT * FROM audio_features WHERE asset_type = 'SONG'"
        params = []
        if query.bpm:
            sql += " AND tempo BETWEEN ? AND ?"
            params.extend([query.bpm - query.bpm_tolerance, query.bpm + query.bpm_tolerance])
        if query.key:
            sql += " AND key = ?"
            params.append(query.key)
        sql += f" LIMIT {query.top_k * 50}"
        sql_results = self.duck_conn.execute(sql, params).df()
        duck_ms = (time.perf_counter() - sql_start) * 1000

        lance_start = time.perf_counter()
        table = self.lance_db.open_table(self.table_name)
        query_vector = self.embedder.encode(query.concept).tolist()
        vector_results = table.search(query_vector).limit(query.top_k * 50).to_pandas()
        lance_ms = (time.perf_counter() - lance_start) * 1000

        merge_start = time.perf_counter()
        merged_df = vector_results.merge(sql_results, on="filename", how="inner")
        
        fused_records = []
        for _, row in merged_df.iterrows():
            physics = TrackPhysics(
                filename=row["filename"],
                dsp_tempo=row.get("tempo"),
                dsp_key=row.get("key"),
                rms_db=row.get("rms_db"),
                crest_factor=row.get("crest_factor"),
                sub_bass_energy=row.get("sub_bass_energy")
            )
            delta = SovereignDelta(
                filename=row["filename"],
                rms_delta=physics.rms_db - (-12.0),
                crest_delta=physics.crest_factor - 5.0,
                sovereign_score=max(0.0, 100.0 - abs(physics.rms_db - (-12.0)) * 10.0),
                grade="A"
            )
            grade_map = {90: "A", 80: "B", 70: "C"}
            delta.grade = next((g for limit, g in grade_map.items() if delta.sovereign_score >= limit), "D")
            fused_records.append(CollisionRecord(
                filename=row["filename"], physics=physics, delta=delta, vector_score=row.get("_distance")
            ))
        merge_ms = (time.perf_counter() - merge_start) * 1000

        return CollisionResult(
            query_concept=query.concept, query_bpm=query.bpm, query_key=query.key,
            level=3, records=fused_records[:query.top_k], total_found=len(fused_records),
            duck_ms=duck_ms, lance_ms=lance_ms, merge_ms=merge_ms
        )

@ray.remote(num_cpus=1)
class SampleOrchestrator:
    def __init__(self, duckdb_path: str, lancedb_path: str):
        import duckdb, lancedb
        from sentence_transformers import SentenceTransformer
        self.duck_conn = duckdb.connect(duckdb_path, read_only=True)
        self.lance_db = lancedb.connect(lancedb_path)
        self.table_name = "audio_vibe_gpu"
        self.embedder = SentenceTransformer("all-MiniLM-L6-v2")

    def run_collision_query(self, query: Level3Query) -> CollisionResult:
        import time
        sql_start = time.perf_counter()
        sql = "SELECT * FROM audio_features WHERE asset_type = 'SAMPLE'"
        params = []
        if query.drum_type:
            sql += " AND drum_type = ?"
            params.append(query.drum_type)
        sql += f" LIMIT {query.top_k * 50}"
        sql_results = self.duck_conn.execute(sql, params).df()
        duck_ms = (time.perf_counter() - sql_start) * 1000

        lance_start = time.perf_counter()
        table = self.lance_db.open_table(self.table_name)
        query_vector = self.embedder.encode(query.concept).tolist()
        vector_results = table.search(query_vector).limit(query.top_k * 50).to_pandas()
        lance_ms = (time.perf_counter() - lance_start) * 1000

        merge_start = time.perf_counter()
        merged_df = vector_results.merge(sql_results, on="filename", how="inner")
        
        fused_records = []
        for _, row in merged_df.iterrows():
            physics = TrackPhysics(
                filename=row["filename"],
                dsp_tempo=None,
                dsp_key=None,
                rms_db=row.get("rms_db"),
                crest_factor=row.get("crest_factor"),
                sub_bass_energy=row.get("sub_bass_energy")
            )
            delta = SovereignDelta(
                filename=row["filename"],
                rms_delta=physics.rms_db - (-12.0),
                crest_delta=physics.crest_factor - 5.0,
                sovereign_score=max(0.0, 100.0 - abs(physics.rms_db - (-12.0)) * 10.0),
                grade="A"
            )
            grade_map = {90: "A", 80: "B", 70: "C"}
            delta.grade = next((g for limit, g in grade_map.items() if delta.sovereign_score >= limit), "D")
            fused_records.append(CollisionRecord(
                filename=row["filename"], physics=physics, delta=delta, vector_score=row.get("_distance")
            ))
        merge_ms = (time.perf_counter() - merge_start) * 1000

        return CollisionResult(
            query_concept=query.concept, query_bpm=None, query_key=None,
            level=3, records=fused_records[:query.top_k], total_found=len(fused_records),
            duck_ms=duck_ms, lance_ms=lance_ms, merge_ms=merge_ms
        )

In [1]:
import json
print("📡 Starting local Ray cluster...")
ray.init(ignore_reinit_error=True, log_to_driver=False)

print("⚙️ Initializing SongOrchestrator and SampleOrchestrator Actors...")
song_orch = SongOrchestrator.remote(DUCKDB_PATH, LANCEDB_PATH)
sample_orch = SampleOrchestrator.remote(DUCKDB_PATH, LANCEDB_PATH)

# Wait for actors to initialize
ray.get(song_orch.run_collision_query.remote(Level3Query(bpm=129, key="G#", top_k=1, concept="test")))
ray.get(sample_orch.run_collision_query.remote(Level3Query(drum_type="KICK", top_k=1, concept="test")))
print("✅ Both Ray Orchestrators Booted and Ready!")

# Run targeted song query
song_q = Level3Query(bpm=129, key="G#", top_k=3, concept="Renegade Master")
print(f"\n🚀 QUERYING SONGS: '{song_q.concept}' (BPM: {song_q.bpm}, Key: {song_q.key})")
res_song = ray.get(song_orch.run_collision_query.remote(song_q))
print(f"⏱️ Response Time: {res_song.duck_ms + res_song.lance_ms + res_song.merge_ms:.2f} ms")
print(f"📊 Fused Songs Found: {res_song.total_found}")
for r in res_song.records:
    print(f"   - {r.filename} (Score: {r.delta.sovereign_score:.1f} | Key: {r.physics.dsp_key} | BPM: {r.physics.dsp_tempo:.1f})")

# Run targeted sample query
sample_q = Level3Query(drum_type="BASS", top_k=3, concept="Synth Lead Loop")
print(f"\n🚀 QUERYING SAMPLES: '{sample_q.concept}' (Type: {sample_q.drum_type})")
res_sample = ray.get(sample_orch.run_collision_query.remote(sample_q))
print(f"⏱️ Response Time: {res_sample.duck_ms + res_sample.lance_ms + res_sample.merge_ms:.2f} ms")
print(f"📊 Fused Samples Found: {res_sample.total_found}")
for r in res_sample.records:
    print(f"   - {r.filename} (Score: {r.delta.sovereign_score:.1f} | RMS: {r.physics.rms_db:.2f} dB)")

ray.shutdown()
print("\n🏁 Ray cluster shut down.")

📡 Starting local Ray cluster...


NameError: name 'ray' is not defined

## Universal Layout-Driven Memory Engine

Below, we implement the layout-driven engine using DuckDB and PyArrow for zero-copy JSON parsing and LanceDB for O(1) in-memory vector splicing.

In [6]:
import pyarrow as pa
import pyarrow.json

# Connect to database for layout configuration
conn_layout = duckdb.connect(DUCKDB_PATH)

# Initialize registry and lookup structures in DuckDB if they don't exist
conn_layout.execute("DROP TABLE IF EXISTS layout_registry")
conn_layout.execute("""
    CREATE TABLE layout_registry (
        layout_id VARCHAR PRIMARY KEY,
        primary_key VARCHAR,
        requires_vector BOOLEAN,
        join_target VARCHAR,
        feature_count INTEGER
    )
""")

# Insert a structural definition as a row
conn_layout.execute("""
    INSERT INTO layout_registry VALUES 
    ('spotify_ableton_sync', 'filename', true, 'audio_features', 46)
""")

# Ensure physical_disk_mapping view exists mapping filename/filepath to asset_id / absolute_path
conn_layout.execute("DROP VIEW IF EXISTS physical_disk_mapping")
conn_layout.execute("""
    CREATE VIEW physical_disk_mapping AS 
    SELECT filename AS asset_id, filepath AS absolute_path 
    FROM audio_features
""")

class UniversalEngine:
    def __init__(self, duckdb_conn, lancedb_table):
        self.db = duckdb_conn
        self.vector_db = lancedb_table

    def execute_payload(self, layout_id: str, incoming_json_data: str):
        # 1. Zero-copy lookup of structural rules from the layout table
        rules = self.db.execute(
            "SELECT * FROM layout_registry WHERE layout_id = ?", [layout_id]
        ).fetchone()
        
        if not rules:
            raise ValueError(f"Layout '{layout_id}' is unregistered in the matrix.")
            
        _, pk_col, requires_vector, join_target, feature_count = rules

        # 2. Parse raw input JSON straight to C++ memory using an Arrow buffer
        buffer = pa.py_buffer(incoming_json_data.encode('utf-8'))
        incoming_table = pa.json.read_json(buffer)
        
        # Register incoming table in DuckDB context dynamically
        self.db.register("incoming_table", incoming_table)
        
        # 3. Dynamic Execution Query Generation
        dynamic_sql = f"""
            SELECT inc.*, target.rms_db, target.crest_factor, target.bass_energy, paths.absolute_path
            FROM incoming_table AS inc
            JOIN {join_target} AS target ON target.{pk_col} = inc.{pk_col}
            JOIN physical_disk_mapping AS paths ON paths.asset_id = target.{pk_col}
        """
        
        start_sql = time.perf_counter()
        fused_matrix = self.db.execute(dynamic_sql).to_arrow_table()
        sql_ms = (time.perf_counter() - start_sql) * 1000

        vector_ms = 0.0
        vector_results = None
        
        # 4. In-Memory Vector Splice (If the layout calls for embeddings)
        if requires_vector and len(fused_matrix) > 0:
            vector_ids = fused_matrix[pk_col].to_pylist()
            
            start_vec = time.perf_counter()
            id_list_str = ", ".join(["'" + x.replace("'", "''") + "'" for x in vector_ids])
            vector_results = self.vector_db.to_lance().to_table(
                filter=f"{pk_col} IN ({id_list_str})"
            )
            vector_ms = (time.perf_counter() - start_vec) * 1000
            
            # Combine the tabular metadata and vectors natively at Arrow C++ speed
            return self.ship_to_cpp_core(fused_matrix, vector_results, sql_ms, vector_ms)

        return self.ship_to_cpp_core(fused_matrix, None, sql_ms, vector_ms)

    def ship_to_cpp_core(self, metadata_table, vector_table, sql_ms, vector_ms):
        print("*** UNIVERSAL ENGINE PAYLOAD EXECUTED ***")
        print("-" * 60)
        print(f"Tabular Join Latency  : {sql_ms:.2f} ms")
        print(f"Vector Splice Latency : {vector_ms:.2f} ms")
        print(f"Row count returned    : {len(metadata_table)}")
        
        print("\nTabular Fields (Arrow Column Names):")
        print(metadata_table.schema.names)
        
        if vector_table is not None:
            print("\nVector Table Fields (Arrow Column Names):")
            print(vector_table.schema.names)
            print(f"Vector dimensions: {len(vector_table['vector'][0]) if len(vector_table) > 0 else 0}")
            
            vec_chunk = vector_table['vector'].chunk(0) if hasattr(vector_table['vector'], 'chunk') else vector_table['vector']
            vec_buffer = vec_chunk.buffers()[1] if hasattr(vec_chunk, 'buffers') else None
            if vec_buffer:
                print(f"Arrow Vector buffer size: {vec_buffer.size} bytes (fully aligned in C++ memory)")
        
        return metadata_table, vector_table

# Setup LanceDB Table
ldb = lancedb.connect(LANCEDB_PATH)
vector_table = ldb.open_table("audio_vibe_gpu")

# Instantiate Engine
engine = UniversalEngine(conn_layout, vector_table)

# Create sample payload
sample_filenames = [
    "63. Meduza - No Sleep (Extended Mix).mp3",
    "RUZE, Chesster - Just Be Good (Original Mix) - www.djsoundtop.com.flac",
    "Enzo Siragusa - Kilimanjaro Sound (Original Mix) - www.djsoundtop.com.flac"
]
incoming_data = "\n".join([json.dumps({"filename": name}) for name in sample_filenames])

# Execute payload
metadata_table, vector_table = engine.execute_payload("spotify_ableton_sync", incoming_data)

# Close layout DuckDB connection
conn_layout.close()

*** UNIVERSAL ENGINE PAYLOAD EXECUTED ***
------------------------------------------------------------
Tabular Join Latency  : 8.14 ms
Vector Splice Latency : 53.38 ms
Row count returned    : 3

Tabular Fields (Arrow Column Names):
['filename', 'rms_db', 'crest_factor', 'bass_energy', 'absolute_path']

Vector Table Fields (Arrow Column Names):
['vector', 'filename', 'filepath', 'source_type', 'drum_type', 'folder']
Vector dimensions: 384
